# 清洗数据导出结果验证

重新读取四张 Parquet 表，验证文件存在性、表结构、主键、字段类型和关键清洗结果。


## 1. 设置目录并查看文件大小

本节检查：

1. 导入 Path 和 Pandas；
2. 指定清洗数据目录 data/processed；
3. 检查四个Parquet文件是否存在，并把字节数换算成MB。

file_path.exists() 用于判断文件是否存在。file_path.stat().st_size 返回文件的字节数。除以 1024 的平方，就能换算成MB。

这一步只检查文件，不读取完整数据。


In [2]:
from pathlib import Path

import pandas as pd


def find_project_root(start: Path | None = None) -> Path:
    """从当前目录向上定位作品集根目录。"""
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "Python").is_dir() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("未找到项目根目录，请从 KuaiRand_Pure 目录或其子目录运行。")


PROJECT_ROOT = find_project_root()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MYSQL_IMPORT_DIR = PROJECT_ROOT / "data" / "mysql_import"

file_names = [
    "log_standard_clean.parquet",
    "log_random_clean.parquet",
    "user_features_clean.parquet",
    "video_features_basic_clean.parquet",
]

for file_name in file_names:
    file_path = PROCESSED_DIR / file_name
    if not file_path.exists():
        raise FileNotFoundError(f"缺少清洗文件：{file_path}")
    print(f"{file_name}: {file_path.stat().st_size / 1024**2:.2f} MB")


log_standard_clean.parquet: 33.72 MB
log_random_clean.parquet: 27.39 MB
user_features_clean.parquet: 0.62 MB
video_features_basic_clean.parquet: 0.26 MB


## 2. 重新读取并验证标准推荐表

pd.read_parquet() 从硬盘重新读取清洗表，这可以证明数据不只存在于原notebook的内存中。

本部分核对：

- 表形状是否为1,414,622行、25列；
- 日期范围和数据来源是否正确；
- is_rand 是否全部为0；
- date_clean 是否存在转换失败。

assert(条件)，这个断言，用来强制检查数据是否符合预期。如果条件为 False，程序会报 AssertionError；如果为 True，就安静通过，不显示结果。


In [2]:
standard_check = pd.read_parquet(
    processed_dir
    / "log_standard_clean.parquet"
)

print("表形状：", standard_check.shape)

print("日期范围：")
print(
    standard_check["date_clean"].min(),
    standard_check["date_clean"].max()
)

print("数据来源分布：")
print(
    standard_check["log_source"]
    .value_counts(dropna=False)
)

print("is_rand分布：")
print(
    standard_check["is_rand"]
    .value_counts(dropna=False)
)

print("字段类型：")
print(standard_check.dtypes)

assert standard_check.shape == (
    1414622,
    25
)

assert (
    standard_check["is_rand"] == 0
).all()

assert (
    standard_check["date_clean"]
    .notna()
    .all()
)

print("标准推荐表验证通过")


表形状： (1414622, 25)
日期范围：
2022-04-09 00:00:00 2022-05-08 00:00:00
数据来源分布：
log_source
standard_0408_0421    1125503
standard_0422_0508     289119
Name: count, dtype: int64
is_rand分布：
is_rand
0    1414622
Name: count, dtype: int64
字段类型：
user_id                         int64
video_id                        int64
date                            int64
hourmin                         int64
time_ms                         int64
is_click                        int64
is_like                         int64
is_follow                       int64
is_comment                      int64
is_forward                      int64
is_hate                         int64
long_view                       int64
play_time_ms                    int64
duration_ms                     int64
profile_stay_time               int64
comment_stay_time               int64
is_profile_enter                int64
is_rand                         int64
tab                             int64
is_duration_missing              int8
durati

## 3. 重新读取并验证随机推荐表

随机推荐表用于与相同日期范围内的标准推荐记录进行描述性比较。

本部分核对：

- 表形状是否为1,186,049行、25列；
- 日期是否全部有效；
- is_rand 是否全部为1。

这里验证 is_rand，是为了防止标准推荐与随机推荐在导出时被混淆。


In [6]:
random_check = pd.read_parquet(
    processed_dir
    / "log_random_clean.parquet"
)

print("表形状：", random_check.shape)

print("日期范围：")
print(
    random_check["date_clean"].min(),
    "-",
    random_check["date_clean"].max()
)

print("is_rand分布：")
print(
    random_check["is_rand"]
    .value_counts(dropna=False)
)

assert random_check.shape == (
    1186049,
    25
)

assert (
    random_check["is_rand"] == 1
).all()

assert (
    random_check["date_clean"]
    .notna()
    .all()
)

print("随机推荐表验证通过")


表形状： (1186049, 25)
日期范围：
2022-04-22 00:00:00 - 2022-05-08 00:00:00
is_rand分布：
is_rand
1    1186049
Name: count, dtype: int64
随机推荐表验证通过


## 4. 重新读取并验证用户特征表

用户特征表是一张用户维度表，每个 user_id 应当只出现一次。

本部分核对：

- 表形状是否为27,285行、32列；
- user_id 是否保持唯一；
- is_live_streamer_clean 中是否保留21,127个缺失值。

这些缺失值是由原字段中的-124转换而来，表示未知状态，而不是“非直播用户”。


In [4]:
user_check = pd.read_parquet(
    processed_dir
    / "user_features_clean.parquet"
)

print("表形状：", user_check.shape)

print(
    "user_id重复数量：",
    user_check["user_id"]
    .duplicated()
    .sum()
)

print("直播用户清洗字段分布：")
print(
    user_check["is_live_streamer_clean"]
    .value_counts(dropna=False)
)

assert user_check.shape == (
    27285,
    32
)

assert user_check["user_id"].is_unique

assert (
    user_check["is_live_streamer_clean"]
    .isna()
    .sum()
    == 21127
)

print("用户特征表验证通过")


表形状： (27285, 32)
user_id重复数量： 0
直播用户清洗字段分布：
is_live_streamer_clean
<NA>    21127
1        6158
Name: count, dtype: Int64
用户特征表验证通过


## 5. 重新读取并验证视频基础表

视频基础表是一张视频维度表，每个 video_id 应当只出现一次。

本部分核对：

- 表形状是否为7,583行、17列；
- video_id 是否唯一；
- 视频时长缺失标记是否为239条；
- music_type_clean 是否为 string、没有缺失且包含203条 UNKNOWN；
- video_duration_seconds 是否等于 video_duration / 1000；
- tag_clean 是否已经没有缺失值；
- upload_date_clean 是否全部转换成功；
- 新增清洗字段的数据类型是否被Parquet正确保留。


In [5]:
video_check = pd.read_parquet(
    processed_dir
    / "video_features_basic_clean.parquet"
)

print("表形状：", video_check.shape)

print(
    "video_id重复数量：",
    video_check["video_id"]
    .duplicated()
    .sum()
)

print(
    "时长缺失标记数量：",
    video_check["is_duration_missing"]
    .sum()
)

print(
    "音乐类型清洗后缺失数量：",
    video_check["music_type_clean"]
    .isna()
    .sum()
)

print(
    "音乐类型UNKNOWN数量：",
    video_check["music_type_clean"]
    .eq("UNKNOWN")
    .sum()
)

print(
    "标签清洗后缺失数量：",
    video_check["tag_clean"]
    .isna()
    .sum()
)

print("新增字段类型：")
print(
    video_check[[
        "upload_date_clean",
        "is_duration_missing",
        "video_duration_seconds",
        "music_type_clean",
        "tag_clean"
    ]].dtypes
)

assert video_check.shape == (
    7583,
    17
)

assert video_check["video_id"].is_unique

assert (
    video_check["is_duration_missing"]
    .sum()
    == 239
)

assert str(video_check["music_type_clean"].dtype).startswith("string")

assert (
    video_check["music_type_clean"]
    .isna()
    .sum()
    == 0
)

assert (
    video_check["music_type_clean"]
    .eq("UNKNOWN")
    .sum()
    == 203
)

assert video_check["video_duration_seconds"].equals(
    video_check["video_duration"] / 1000
)

assert (
    video_check["tag_clean"]
    .isna()
    .sum()
    == 0
)

assert (
    video_check["upload_date_clean"]
    .notna()
    .all()
)

print("视频基础表验证通过")


表形状： (7583, 17)
video_id重复数量： 0
时长缺失标记数量： 239
音乐类型清洗后缺失数量： 0
音乐类型UNKNOWN数量： 203
标签清洗后缺失数量： 0
新增字段类型：
upload_date_clean          datetime64[us]
is_duration_missing                  int8
video_duration_seconds            float64
music_type_clean                   string
tag_clean                          string
dtype: object
视频基础表验证通过


## 6. 验证后的结果
四张清洗表已经成功保存，重新读取后的行列数、主键、日期和关键缺失标记均符合预期。

后续核心分析将从 data/processed 中读取这四张Parquet文件，不再依赖之前三个清洗notebook里的内存变量。
